# Object Recognition using ResNet50
This notebook demonstrates image classification on the CIFAR-10 dataset using a pre-trained ResNet50 model and transfer learning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import datasets, layers, models, optimizers
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.utils import to_categorical

## Data Loading and Preprocessing
We use the CIFAR-10 dataset, which consists of 60,000 32x32 color images in 10 classes.

In [ ]:
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

# Normalize pixel values
x_train, x_test = x_train / 255.0, x_test / 255.0

# Convert labels to one-hot encoding if using categorical_crossentropy
# Or keep as is for sparse_categorical_crossentropy
print(f"Train data shape: {x_train.shape}")
print(f"Test data shape: {x_test.shape}")

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i])
    plt.xlabel(class_names[y_train[i][0]])
plt.show()

## Build the Model (Transfer Learning)
ResNet50 requires a minimum input size of 32x32, but it performs better if we upscale the CIFAR images.

In [ ]:
convolutional_base = ResNet50(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
convolutional_base.trainable = False  # Freeze the base

model = models.Sequential([
    convolutional_base,
    layers.Flatten(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.BatchNormalization(),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## Model Training

In [ ]:
history = model.fit(x_train, y_train, epochs=10, 
                    validation_data=(x_test, y_test))

## Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f'Test Accuracy: {test_acc}')

plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.legend(loc='lower right')
plt.show()